# Smart Multimodal Mobility — Modeling Deliverable (Check)

## ADEME context and objective

This notebook formalizes a delivery route optimization problem for CesiCDP's answer to ADEME's call for projects on sustainable mobility.

We study a **fleet routing problem on a road network** where travel times depend on traffic conditions over time. The objective is to design delivery routes that are economically efficient and less polluting by reducing total route duration.


## 1) Problem statement (formal view)

Given:

- A directed graph \(G=(V,A)\) representing the road network.
- A depot node \(0\in V\).
- A set of customers \(C = V\setminus\{0\}\).
- A fleet of \(k\) identical trucks.
- Service demand \(q_i\) for each customer \(i\in C\).
- Truck capacity \(Q\).
- Time-dependent travel duration \(\tau_{ij}(t)\) on each arc \((i,j)\in A\).

Basic version requirements:

1. Build routes that start and end at the depot.
2. Visit each selected customer exactly once.
3. Respect truck capacity.
4. Minimize total travel (and service) duration.

Advanced version additional constraints:

1. Time window \([a_i,b_i]\) for each customer service.
2. Delivery outside \([a_i,b_i]\) is forbidden.
3. Waiting on site (or before service) is allowed when arriving earlier than \(a_i\).


## 2) Data representation

A practical instance can be represented by:

- `nodes`: list of vertices with `id`, `type` (`depot` or `customer`).
- `arcs`: adjacency list with distance and time-dependent travel function parameters.
- `demands`: vector \((q_i)_{i\in C}\).
- `fleet`: number of trucks \(k\), common capacity \(Q\).
- `service_times`: \(s_i\) for all customers.
- `time_windows`: \((a_i,b_i)\) for all customers (advanced version).

For large-scale instances (thousands of customers), sparse graph storage and compressed travel-time tables by time slots are recommended.


## 3) Link with classical OR problems and complexity

The model is a variant of:

- **CVRP** (Capacitated Vehicle Routing Problem) for the basic fleet + capacity constraints.
- **VRPTW** (Vehicle Routing Problem with Time Windows) for the advanced version.
- **Time-Dependent VRP** due to \(\tau_{ij}(t)\).

Complexity argument:

- The problem is NP-hard because CVRP is NP-hard.
- A reduction is immediate: if we set a single truck with large capacity and no time windows, the model contains TSP as a special case.
- Therefore, the decision version is NP-complete (in NP and NP-hard) under standard bounded encoding assumptions.

This justifies exact + heuristic/approximate approaches for large instances.


## 4) Integer Linear Programming (ILP) model

### 4.1 Sets and indices

- \(V = \{0\}\cup C\): depot + customers.
- \(K = \{1,\dots,k\}\): trucks.
- \(A \subseteq V\times V\): feasible directed arcs, \(i\neq j\).

### 4.2 Parameters

- \(q_i\): demand of customer \(i\in C\).
- \(Q\): truck capacity.
- \(c_{ij}\): travel cost/time proxy on arc \((i,j)\) (piecewise approximation of time-dependent durations).
- \(s_i\): service duration at customer \(i\).
- \([a_i,b_i]\): service time window at customer \(i\) (advanced version).
- \(M\): sufficiently large constant.

### 4.3 Decision variables

- \(x_{ij}^u \in \{0,1\}\): 1 if truck \(u\) travels directly from \(i\) to \(j\), else 0.
- \(y_i^u \in \{0,1\}\): 1 if customer \(i\) is served by truck \(u\), else 0.
- \(T_i^u \ge 0\): service start time of truck \(u\) at node \(i\).
- \(w_i^u \ge 0\): waiting time before service at customer \(i\) (advanced version).
- \(L_i^u\): load after visiting node \(i\) for truck \(u\) (MTZ-like capacity tracking).

### 4.4 Objective

Minimize total route duration (travel + waiting + service):

\[
\min \sum_{u\in K}\sum_{(i,j)\in A} c_{ij}x_{ij}^u + \sum_{u\in K}\sum_{i\in C} w_i^u + \sum_{u\in K}\sum_{i\in C} s_i y_i^u
\]

(Equivalent formulations may move service terms to constants when all customers are mandatory.)

### 4.5 Constraints

#### (a) Customer assignment (exactly once)

\[
\sum_{u\in K} y_i^u = 1,\quad \forall i\in C
\]

#### (b) Flow conservation and visit consistency

\[
\sum_{j:(i,j)\in A} x_{ij}^u = y_i^u,\quad \forall u\in K,\forall i\in C
\]
\[
\sum_{j:(j,i)\in A} x_{ji}^u = y_i^u,\quad \forall u\in K,\forall i\in C
\]

#### (c) Depot departure/return per used truck

\[
\sum_{j:(0,j)\in A} x_{0j}^u \le 1,\quad \forall u\in K
\]
\[
\sum_{i:(i,0)\in A} x_{i0}^u \le 1,\quad \forall u\in K
\]
\[
\sum_{j:(0,j)\in A} x_{0j}^u = \sum_{i:(i,0)\in A} x_{i0}^u,\quad \forall u\in K
\]

#### (d) Capacity constraints

\[
0 \le L_i^u \le Q,\quad \forall u\in K,\forall i\in V
\]
\[
L_j^u \ge L_i^u + q_j - M(1-x_{ij}^u),\quad \forall u\in K,\forall (i,j)\in A,\ j\in C
\]
\[
q_i y_i^u \le L_i^u \le Qy_i^u + Q(1-y_i^u),\quad \forall u\in K,\forall i\in C
\]

#### (e) Time propagation (time-dependent proxy)

\[
T_j^u \ge T_i^u + s_i + c_{ij} + w_j^u - M(1-x_{ij}^u),\quad \forall u\in K,\forall (i,j)\in A,\ j\in C
\]

#### (f) Time windows and no out-of-window delivery (advanced version)

\[
a_i y_i^u \le T_i^u \le b_i y_i^u + M(1-y_i^u),\quad \forall u\in K,\forall i\in C
\]
\[
w_i^u \ge a_i - T_i^u,\quad w_i^u \ge 0,\quad \forall u\in K,\forall i\in C
\]

These constraints force service within \([a_i,b_i]\) and allow waiting if a vehicle arrives early.

#### (g) Domains

\[
x_{ij}^u \in \{0,1\},\ y_i^u\in\{0,1\},\ T_i^u\ge 0,\ w_i^u\ge0
\]

This ILP provides a rigorous baseline model to be implemented and refined in subsequent project phases.


## 5) Notes for future implementation and experimentation

- For several thousand cities, exact ILP solving may become intractable.
- A practical pipeline is typically:
  1. preprocessing and graph sparsification,
  2. construction heuristic (e.g., savings/insertion),
  3. improvement metaheuristics (e.g., local search, ALNS),
  4. exact polishing on restricted neighborhoods.
- The experimental study should report solution quality, runtime distribution, scalability, and sensitivity to traffic/time-window tightness.


## 6) Bibliographic references

1. Toth, P., & Vigo, D. (2014). *Vehicle Routing: Problems, Methods, and Applications* (2nd ed.). SIAM.
2. Desaulniers, G., Desrosiers, J., & Solomon, M. M. (Eds.). (2005). *Column Generation*. Springer. (VRPTW formulations)
3. Solomon, M. M. (1987). Algorithms for the Vehicle Routing and Scheduling Problems with Time Window Constraints. *Operations Research*, 35(2), 254–265.
4. Lenstra, J. K., & Rinnooy Kan, A. H. G. (1981). Complexity of Vehicle Routing and Scheduling Problems. *Networks*, 11(2), 221–227.
5. Gendreau, M., Ghiani, G., & Guerriero, E. (2015). Time-Dependent Routing Problems: A Review. *Computers & Operations Research*, 64, 189–197.
